# Telegram Public Channel Snowball Scraper

Pilot collection notebook for JudolLens thesis data. The goal is to collect auditable Telegram public-channel messages for manual annotation, not to create final labels automatically.

**Method:** seed public channels -> crawl recent messages -> discover linked/forwarded channels -> enrich rows with candidate flags -> export raw and annotation queue files.

**Important:** use only public channels. Do not collect private chats, private groups, media files, phone numbers, or direct personal identifiers unless a thesis ethics review explicitly approves it.

## 1. Install Dependencies

Run this once per environment. Telethon uses Telegram's MTProto API as a user client, so the first login may ask for phone, OTP, and two-factor password.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "telethon": "telethon",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print(f"Installed missing packages: {missing}")
else:
    print("Notebook dependencies already available.")


Notebook dependencies already available.


## 2. Configuration

Put credentials in the repository `.env` file, not in this notebook:

```env
TELEGRAM_API_ID=123456
TELEGRAM_API_HASH=replace-with-your-telegram-api-hash
```

Get credentials from <https://my.telegram.org>. Keep `RUN_COLLECTION = False` until the configuration summary looks correct.

In [5]:
from pathlib import Path
from datetime import datetime, timezone
import os
import re
import json
import hashlib
import sqlite3
import asyncio
from urllib.parse import urlparse

CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / "notebooks").exists() and (CURRENT_DIR / "docs").exists():
    PROJECT_ROOT = CURRENT_DIR
elif CURRENT_DIR.name == "notebooks" and (CURRENT_DIR.parent / "docs").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

ENV_PATH = PROJECT_ROOT / ".env"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_FILE = OUTPUT_DIR / "telegram_snowball_messages.sqlite"
RAW_OUTPUT = OUTPUT_DIR / "telegram_messages_raw.csv"
ANNOTATION_OUTPUT = OUTPUT_DIR / "telegram_messages_annotation_queue.csv"
CHANNEL_OUTPUT = OUTPUT_DIR / "telegram_channel_provenance.csv"

RUN_COLLECTION = True  # Set True only after credentials, seeds, and limits are reviewed.
SESSION_NAME = str(PROJECT_ROOT / ".telegram_scraper_session")

MAX_DEPTH = 1
MAX_MESSAGES_PER_CHANNEL = 300
MIN_WORDS = 4
MIN_MEMBERS = 50
SLEEP_SECONDS = 4
INCLUDE_NEGATIVE_SEEDS = True

PLACEHOLDER_VALUES = {"", "replace-with-your-telegram-api-hash", "YOUR_API_HASH_HERE", "YOUR_API_ID_HERE"}


def clean_env_value(value: str | None) -> str:
    value = (value or "").strip().strip('"').strip("'")
    return "" if value in PLACEHOLDER_VALUES else value


def load_simple_env(env_path: Path) -> dict[str, str]:
    if not env_path.exists():
        return {}
    loaded = {}
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = clean_env_value(value)
        if key and value and not clean_env_value(os.environ.get(key)):
            os.environ[key] = value
        loaded[key] = value
    return loaded

LOADED_ENV = load_simple_env(ENV_PATH)
TELEGRAM_API_ID_RAW = clean_env_value(os.getenv("TELEGRAM_API_ID"))
TELEGRAM_API_HASH = clean_env_value(os.getenv("TELEGRAM_API_HASH"))
TELEGRAM_API_ID = int(TELEGRAM_API_ID_RAW) if TELEGRAM_API_ID_RAW.isdigit() else 0

SEED_CHANNELS = [
    {
        "channel_username": "SPINWIN77GACOR",
        "seed_group": "judol_candidate",
        "seed_type": "manual_public_channel",
        "seed_reason": "explicit judol-style public channel seed",
        "paper_basis": "Perdana-2024 keyword seed + Telegram platform extension",
    },
    {
        "channel_username": "vvip_888slot",
        "seed_group": "judol_candidate",
        "seed_type": "manual_public_channel",
        "seed_reason": "slot/freebet promotion style seed",
        "paper_basis": "Ammar-2025 keyword/contact heuristic adapted to Telegram",
    },
    {
        "channel_username": "infoslotgacor1st",
        "seed_group": "judol_candidate",
        "seed_type": "manual_public_channel",
        "seed_reason": "explicit slot-gacor public channel seed",
        "paper_basis": "Perdana-2024 keyword seed + Telegram platform extension",
    },
]

NEGATIVE_SEED_CHANNELS = [
    {
        "channel_username": "kompascom",
        "seed_group": "control_negative",
        "seed_type": "manual_public_channel",
        "seed_reason": "large Indonesian news channel for hard-negative/control text",
        "paper_basis": "negative control sampling",
    },
    {
        "channel_username": "BBCIndonesia",
        "seed_group": "control_negative",
        "seed_type": "manual_public_channel",
        "seed_reason": "Indonesian news channel for clean/control samples",
        "paper_basis": "negative control sampling",
    },
]

ALL_SEEDS = SEED_CHANNELS + (NEGATIVE_SEED_CHANNELS if INCLUDE_NEGATIVE_SEEDS else [])

print({
    "project_root": str(PROJECT_ROOT),
    "env_path": str(ENV_PATH),
    "env_file_present": ENV_PATH.exists(),
    "loaded_env_keys": sorted(k for k in LOADED_ENV if k.startswith("TELEGRAM_")),
    "telegram_api_id_present": TELEGRAM_API_ID > 0,
    "telegram_api_hash_present": bool(TELEGRAM_API_HASH),
    "run_collection": RUN_COLLECTION,
    "seed_count": len(ALL_SEEDS),
    "db_file": str(DB_FILE),
    "raw_output": str(RAW_OUTPUT),
})

{'project_root': 'd:\\judol_detection', 'env_path': 'd:\\judol_detection\\.env', 'env_file_present': True, 'loaded_env_keys': ['TELEGRAM_API_HASH', 'TELEGRAM_API_ID'], 'telegram_api_id_present': True, 'telegram_api_hash_present': True, 'run_collection': True, 'seed_count': 5, 'db_file': 'd:\\judol_detection\\data\\interim\\telegram_snowball_messages.sqlite', 'raw_output': 'd:\\judol_detection\\data\\interim\\telegram_messages_raw.csv'}


## 3. Keyword and Evidence Rules

These rules are **candidate flags**, not final labels. Final labels should be assigned manually in the exported annotation queue.

In [6]:
BASELINE_KEYWORDS = [
    "slot", "gacor", "judi", "togel", "casino", "kasino", "poker", "betting", "taruhan",
    "bonus", "deposit", "depo", "wd", "withdraw", "maxwin", "jackpot", "scatter", "rtp",
    "freebet", "cashback", "rollingan", "new member", "member baru", "daftar", "link alternatif",
    "pragmatic", "pg soft", "habanero", "joker123", "idn poker", "idn slot",
]

CONTACT_PATTERNS = [
    r"https?://", r"www\.", r"t\.me/", r"telegram\.me/", r"telegram", r"whatsapp",
    r"\bwa\b", r"bit\.ly", r"linktr\.ee", r"rdct\.cc", r"cutt\.ly", r"tinyurl\.com",
]

INDONESIAN_HINT_WORDS = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "ini", "itu", "tidak", "bisa",
    "ada", "karena", "kalau", "sudah", "belum", "hari", "sekarang", "gratis", "daftar",
}

HOMOGLYPH_MAP = str.maketrans({
    "?": "o", "?": "O", "?": "a", "?": "A", "?": "e", "?": "E", "?": "i", "?": "I",
    "?": "s", "?": "S", "?": "c", "?": "C", "?": "p", "?": "P", "?": "x", "?": "X",
    "?": "0", "?": "1", "?": "2", "?": "3", "?": "4", "?": "5", "?": "6", "?": "7", "?": "8", "?": "9",
})

KEYWORD_RE = re.compile("|".join(re.escape(k) for k in sorted(BASELINE_KEYWORDS, key=len, reverse=True)), re.IGNORECASE)
CONTACT_RE = re.compile("|".join(CONTACT_PATTERNS), re.IGNORECASE)
CHANNEL_LINK_RE = re.compile(r"(?:https?://)?(?:t\.me|telegram\.me)/([A-Za-z0-9_]{5,32})", re.IGNORECASE)
USERNAME_RE = re.compile(r"(?<!\w)@([A-Za-z0-9_]{5,32})")
ZERO_WIDTH_RE = re.compile(r"[\u200b\u200c\u200d\ufeff]")
SPACED_WORD_RE = re.compile(r"\b(?:[a-zA-Z0-9]\s+){3,}[a-zA-Z0-9]\b")
NUMBER_SUB_RE = re.compile(r"\b\w*[04@1!3$5]\w*\b", re.IGNORECASE)


def normalize_text_for_matching(text: str) -> str:
    compact = ZERO_WIDTH_RE.sub("", text or "")
    compact = compact.translate(HOMOGLYPH_MAP)
    compact = re.sub(r"[\*_`~]", "", compact)
    return compact


def extract_matched_keywords(text: str) -> list[str]:
    normalized = normalize_text_for_matching(text)
    return sorted({m.group(0).lower() for m in KEYWORD_RE.finditer(normalized)})


def extract_channel_links(text: str) -> list[str]:
    found = {m.group(1).lower() for m in CHANNEL_LINK_RE.finditer(text or "")}
    # Username mentions are only used as snowball hints if they appear in a message that already has judol/contact evidence.
    if CONTACT_RE.search(text or "") or KEYWORD_RE.search(normalize_text_for_matching(text or "")):
        found.update(m.group(1).lower() for m in USERNAME_RE.finditer(text or ""))
    return sorted(found)


def is_probably_indonesian(text: str) -> bool:
    tokens = re.findall(r"[a-zA-Z]{2,}", (text or "").lower())
    if not tokens:
        return False
    return bool(set(tokens) & INDONESIAN_HINT_WORDS) or len(tokens) >= 6


def unicode_flags(text: str) -> list[str]:
    flags = []
    value = text or ""
    if ZERO_WIDTH_RE.search(value):
        flags.append("zero_width")
    if value != value.translate(HOMOGLYPH_MAP):
        flags.append("homoglyph")
    if SPACED_WORD_RE.search(value):
        flags.append("spaced_word")
    if NUMBER_SUB_RE.search(value):
        flags.append("number_substitution")
    return sorted(set(flags))


def has_contact_evidence(text: str) -> bool:
    return bool(CONTACT_RE.search(text or ""))


def sha256_text(value: str) -> str:
    return hashlib.sha256((value or "").encode("utf-8")).hexdigest()


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def message_to_row(message, channel_meta: dict, seed_meta: dict, found_from: str) -> dict:
    raw_text = message.message or ""
    normalized = normalize_text_for_matching(raw_text)
    matched_keywords = extract_matched_keywords(raw_text)
    u_flags = unicode_flags(raw_text)
    contact_flag = has_contact_evidence(raw_text)
    candidate_flag_keyword = bool(matched_keywords)
    candidate_flag_unicode = bool(u_flags)
    candidate_score = sum([
        candidate_flag_keyword,
        contact_flag,
        candidate_flag_unicode,
        seed_meta.get("seed_group") == "judol_candidate",
    ])

    return {
        "platform": "telegram",
        "source_type": "public_channel_api",
        "source_seed": seed_meta.get("channel_username", ""),
        "source_group": seed_meta.get("seed_group", ""),
        "seed_type": seed_meta.get("seed_type", ""),
        "seed_reason": seed_meta.get("seed_reason", ""),
        "paper_basis": seed_meta.get("paper_basis", ""),
        "found_from": found_from,
        "channel_username": channel_meta.get("channel_username", ""),
        "channel_title": channel_meta.get("channel_title", ""),
        "channel_id_hash": channel_meta.get("channel_id_hash", ""),
        "channel_member_count": channel_meta.get("member_count"),
        "message_id": getattr(message, "id", None),
        "message_date": getattr(message, "date", None).isoformat() if getattr(message, "date", None) else "",
        "raw_text": raw_text,
        "text_hash": sha256_text(raw_text.lower().strip()),
        "matched_keywords": "|".join(matched_keywords),
        "candidate_flag_keyword": candidate_flag_keyword,
        "candidate_flag_contact": contact_flag,
        "candidate_flag_unicode": candidate_flag_unicode,
        "unicode_obfuscation_flags": "|".join(u_flags),
        "candidate_score": candidate_score,
        "language_hint": "id" if is_probably_indonesian(raw_text) else "unknown",
        "label": "perlu_review",
        "label_source": "unlabeled",
        "notes": "",
        "collected_at": utc_now_iso(),
    }

## 4. SQLite Store

The database lets the crawl resume safely and keeps duplicate messages out of the annotation queue.

In [7]:
def init_db(db_file: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(db_file, timeout=30, check_same_thread=False)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            text_hash TEXT PRIMARY KEY,
            payload_json TEXT NOT NULL,
            inserted_at TEXT NOT NULL
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS channel_queue (
            channel_username TEXT PRIMARY KEY,
            depth INTEGER NOT NULL,
            seed_json TEXT NOT NULL,
            found_from TEXT NOT NULL
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS visited_channels (
            channel_username TEXT PRIMARY KEY,
            depth INTEGER NOT NULL,
            status TEXT NOT NULL,
            message_count INTEGER NOT NULL,
            saved_count INTEGER NOT NULL,
            skipped_count INTEGER NOT NULL,
            discovered_count INTEGER NOT NULL,
            scraped_at TEXT NOT NULL,
            error TEXT
        )
    """)
    conn.commit()
    return conn


def enqueue_channel(conn: sqlite3.Connection, username: str, depth: int, seed_meta: dict, found_from: str) -> None:
    username = (username or "").strip().lstrip("@").lower()
    if not username:
        return
    conn.execute(
        "INSERT OR IGNORE INTO channel_queue VALUES (?,?,?,?)",
        (username, depth, json.dumps(seed_meta, ensure_ascii=False), found_from),
    )
    conn.commit()


def pop_channel(conn: sqlite3.Connection):
    row = conn.execute(
        "SELECT channel_username, depth, seed_json, found_from FROM channel_queue ORDER BY depth, channel_username LIMIT 1"
    ).fetchone()
    if not row:
        return None
    conn.execute("DELETE FROM channel_queue WHERE channel_username=?", (row[0],))
    conn.commit()
    return row[0], row[1], json.loads(row[2]), row[3]


def record_visit(conn: sqlite3.Connection, username: str, depth: int, status: str, stats: dict, error: str = "") -> None:
    conn.execute(
        "INSERT OR REPLACE INTO visited_channels VALUES (?,?,?,?,?,?,?,?,?)",
        (
            username,
            depth,
            status,
            int(stats.get("message_count", 0)),
            int(stats.get("saved_count", 0)),
            int(stats.get("skipped_count", 0)),
            int(stats.get("discovered_count", 0)),
            utc_now_iso(),
            error,
        ),
    )
    conn.commit()


def save_message_row(conn: sqlite3.Connection, row: dict) -> bool:
    try:
        conn.execute(
            "INSERT INTO messages VALUES (?,?,?)",
            (row["text_hash"], json.dumps(row, ensure_ascii=False), utc_now_iso()),
        )
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False


def load_message_rows(conn: sqlite3.Connection) -> list[dict]:
    rows = conn.execute("SELECT payload_json FROM messages").fetchall()
    return [json.loads(row[0]) for row in rows]


def load_channel_provenance(conn: sqlite3.Connection):
    import pandas as pd
    return pd.read_sql_query("SELECT * FROM visited_channels ORDER BY scraped_at DESC", conn)

conn = init_db(DB_FILE)
for seed in ALL_SEEDS:
    enqueue_channel(conn, seed["channel_username"], 0, seed, "seed")

print({
    "db_file": str(DB_FILE),
    "queued_channels": conn.execute("SELECT COUNT(*) FROM channel_queue").fetchone()[0],
    "stored_messages": conn.execute("SELECT COUNT(*) FROM messages").fetchone()[0],
})

{'db_file': 'd:\\judol_detection\\data\\interim\\telegram_snowball_messages.sqlite', 'queued_channels': 5, 'stored_messages': 0}


## 5. Login to Telegram

This cell starts the Telethon client. It will prompt for login only when the local session file is not already authenticated.

In [ ]:
from telethon import TelegramClient, functions
from telethon.errors import ChannelPrivateError, ChatAdminRequiredError, FloodWaitError, UsernameNotOccupiedError

if not TELEGRAM_API_ID or not TELEGRAM_API_HASH:
    print("Telegram credentials are missing. Fill TELEGRAM_API_ID and TELEGRAM_API_HASH in .env before collection.")
else:
    client = TelegramClient(SESSION_NAME, TELEGRAM_API_ID, TELEGRAM_API_HASH)
    if RUN_COLLECTION:
        await client.start()
        me = await client.get_me()
        print(f"Logged in as: {getattr(me, 'first_name', '')} (@{getattr(me, 'username', '')})")
    else:
        print("RUN_COLLECTION is False. Set it to True after reviewing configuration.")

Signed in successfully as Pratama; remember to not break the ToS or you will risk an account ban!
Logged in as: Pratama (@Skysky249)


Server closed the connection: [WinError 10054] An existing connection was forcibly closed by the remote host
Server closed the connection: [WinError 10054] An existing connection was forcibly closed by the remote host
Server closed the connection: [WinError 10054] An existing connection was forcibly closed by the remote host


## 6. Collection Functions

The crawler reads recent public-channel messages, discovers linked/forwarded public channels, and respects `FloodWaitError` by sleeping before continuing.

In [9]:
async def get_channel_metadata(client, username: str) -> dict:
    entity = await client.get_entity(username)
    full = await client(functions.channels.GetFullChannelRequest(entity))
    member_count = getattr(full.full_chat, "participants_count", None) or 0
    return {
        "entity": entity,
        "channel_username": getattr(entity, "username", username) or username,
        "channel_title": getattr(entity, "title", ""),
        "channel_id_hash": sha256_text(str(getattr(entity, "id", ""))),
        "member_count": member_count,
    }


async def scrape_one_channel(client, conn: sqlite3.Connection, username: str, depth: int, seed_meta: dict, found_from: str) -> dict:
    stats = {"message_count": 0, "saved_count": 0, "skipped_count": 0, "discovered_count": 0}
    discovered_channels = set()

    try:
        channel_meta = await get_channel_metadata(client, username)
        if channel_meta["member_count"] and channel_meta["member_count"] < MIN_MEMBERS:
            record_visit(conn, username, depth, "skipped_small_channel", stats)
            return stats

        async for message in client.iter_messages(channel_meta["entity"], limit=MAX_MESSAGES_PER_CHANNEL):
            stats["message_count"] += 1
            raw_text = message.message or ""
            if not raw_text.strip() or len(raw_text.split()) < MIN_WORDS:
                stats["skipped_count"] += 1
                continue
            if not is_probably_indonesian(raw_text):
                stats["skipped_count"] += 1
                continue

            discovered_channels.update(extract_channel_links(raw_text))

            if getattr(message, "fwd_from", None) and getattr(message.fwd_from, "from_id", None):
                try:
                    forwarded_entity = await client.get_entity(message.fwd_from.from_id)
                    forwarded_username = getattr(forwarded_entity, "username", "")
                    if forwarded_username:
                        discovered_channels.add(forwarded_username.lower())
                except Exception:
                    pass

            row = message_to_row(message, channel_meta, seed_meta, found_from)
            if save_message_row(conn, row):
                stats["saved_count"] += 1
            else:
                stats["skipped_count"] += 1

        for discovered in sorted(discovered_channels):
            if depth + 1 <= MAX_DEPTH and not discovered.startswith("telegram"):
                next_seed = dict(seed_meta)
                next_seed["channel_username"] = discovered
                next_seed["seed_type"] = "snowball_public_channel"
                next_seed["seed_reason"] = f"discovered from @{username}"
                enqueue_channel(conn, discovered, depth + 1, next_seed, username)

        stats["discovered_count"] = len(discovered_channels)
        record_visit(conn, username, depth, "ok", stats)
        return stats

    except FloodWaitError as error:
        wait_seconds = int(getattr(error, "seconds", 0)) + 5
        print(f"FloodWaitError on @{username}; sleeping {wait_seconds}s")
        await asyncio.sleep(wait_seconds)
        record_visit(conn, username, depth, "flood_wait", stats, str(error))
        return stats
    except (ChannelPrivateError, UsernameNotOccupiedError, ChatAdminRequiredError) as error:
        record_visit(conn, username, depth, "unavailable", stats, error.__class__.__name__)
        return stats
    except Exception as error:
        record_visit(conn, username, depth, "error", stats, repr(error))
        return stats


async def run_snowball_collection(client, conn: sqlite3.Connection) -> dict:
    total = {"channels": 0, "messages_seen": 0, "messages_saved": 0, "messages_skipped": 0}
    while True:
        item = pop_channel(conn)
        if item is None:
            break
        username, depth, seed_meta, found_from = item
        print(f"@{username} depth={depth} from={found_from}")
        stats = await scrape_one_channel(client, conn, username, depth, seed_meta, found_from)
        total["channels"] += 1
        total["messages_seen"] += stats["message_count"]
        total["messages_saved"] += stats["saved_count"]
        total["messages_skipped"] += stats["skipped_count"]
        await asyncio.sleep(SLEEP_SECONDS)
    return total

## 7. Run Collection

Run this after setting `RUN_COLLECTION = True`. The notebook stores progress in SQLite, so it can be resumed later.

In [10]:
if RUN_COLLECTION and TELEGRAM_API_ID and TELEGRAM_API_HASH:
    collection_summary = await run_snowball_collection(client, conn)
else:
    collection_summary = {"channels": 0, "messages_seen": 0, "messages_saved": 0, "messages_skipped": 0}
    print("Collection skipped. Set RUN_COLLECTION=True and provide Telegram credentials to fetch live data.")

collection_summary

@bbcindonesia depth=0 from=seed
@infoslotgacor1st depth=0 from=seed
@kompascom depth=0 from=seed
@spinwin77gacor depth=0 from=seed
@vvip_888slot depth=0 from=seed
@anellezptt depth=1 from=infoslotgacor1st
@bbcindonesia depth=1 from=bbcindonesia
@beasiswac depth=1 from=infoslotgacor1st
@eldsgroup depth=1 from=infoslotgacor1st
@elodiecc depth=1 from=infoslotgacor1st
@elodieofc depth=1 from=infoslotgacor1st
@elodieofcbot depth=1 from=infoslotgacor1st
@gamblers19 depth=1 from=infoslotgacor1st
@inooui depth=1 from=infoslotgacor1st
@jungstore depth=1 from=infoslotgacor1st
@kedaiiluxe depth=1 from=infoslotgacor1st
@kedailuxe_bot depth=1 from=infoslotgacor1st
@kompascom depth=1 from=kompascom
@luxecatalogue depth=1 from=infoslotgacor1st
@marketwhizz depth=1 from=bbcindonesia
@marveleux_bot depth=1 from=infoslotgacor1st
@mhsastore depth=1 from=infoslotgacor1st
@proofjungs depth=1 from=infoslotgacor1st
@scandalxours depth=1 from=infoslotgacor1st
@supelmans depth=1 from=infoslotgacor1st
@taeheyun

{'channels': 29,
 'messages_seen': 2731,
 'messages_saved': 1403,
 'messages_skipped': 1328}

## 8. Export Raw Data and Annotation Queue

`label` remains `perlu_review`. Sort by candidate evidence first so annotation can start with likely positives, while clean/control rows remain available for false-positive checks.

In [11]:
import pandas as pd

rows = load_message_rows(conn)
df_raw = pd.DataFrame(rows)

if df_raw.empty:
    print("No Telegram messages stored yet.")
    df_raw = pd.DataFrame(columns=[
        "platform", "source_type", "source_seed", "source_group", "channel_username", "message_id",
        "message_date", "raw_text", "matched_keywords", "candidate_flag_keyword", "candidate_flag_contact",
        "candidate_flag_unicode", "unicode_obfuscation_flags", "candidate_score", "label", "label_source",
        "paper_basis", "collected_at",
    ])

sort_cols = [col for col in ["candidate_score", "message_date"] if col in df_raw.columns]
if sort_cols:
    df_raw = df_raw.sort_values(sort_cols, ascending=[False, False][:len(sort_cols)]).reset_index(drop=True)

annotation_columns = [
    "platform", "source_type", "source_seed", "source_group", "channel_username", "channel_title",
    "message_id", "message_date", "raw_text", "matched_keywords", "candidate_flag_keyword",
    "candidate_flag_contact", "candidate_flag_unicode", "unicode_obfuscation_flags", "candidate_score",
    "language_hint", "label", "label_source", "notes", "paper_basis", "found_from", "collected_at",
]
annotation_columns = [col for col in annotation_columns if col in df_raw.columns]
df_annotation = df_raw[annotation_columns].copy()

channel_df = load_channel_provenance(conn)

df_raw.to_csv(RAW_OUTPUT, index=False, encoding="utf-8")
df_annotation.to_csv(ANNOTATION_OUTPUT, index=False, encoding="utf-8")
channel_df.to_csv(CHANNEL_OUTPUT, index=False, encoding="utf-8")

print({
    "raw_rows": len(df_raw),
    "annotation_rows": len(df_annotation),
    "channels_visited": len(channel_df),
    "raw_output": str(RAW_OUTPUT),
    "annotation_output": str(ANNOTATION_OUTPUT),
    "channel_output": str(CHANNEL_OUTPUT),
})

df_annotation.head(10)

{'raw_rows': 1403, 'annotation_rows': 1403, 'channels_visited': 27, 'raw_output': 'd:\\judol_detection\\data\\interim\\telegram_messages_raw.csv', 'annotation_output': 'd:\\judol_detection\\data\\interim\\telegram_messages_annotation_queue.csv', 'channel_output': 'd:\\judol_detection\\data\\interim\\telegram_channel_provenance.csv'}


,platform,source_type,source_seed,source_group,channel_username,channel_title,message_id,message_date,raw_text,matched_keywords,...,candidate_flag_unicode,unicode_obfuscation_flags,candidate_score,language_hint,label,label_source,notes,paper_basis,found_from,collected_at
0,telegram,public_channel_api,infoslotgacor1st,judol_candidate,infoslotgacor1st,INFO SLOT GACOR,8065,2024-11-02T15:22:58+00:00,MULAI SAAT INI PRGMATIC BISA MAIN DI BET 400P ...,pragmatic,...,True,number_substitution,4,id,perlu_review,unlabeled,,Perdana-2024 keyword seed + Telegram platform ...,seed,2026-07-09T17:14:22+00:00
1,telegram,public_channel_api,infoslotgacor1st,judol_candidate,infoslotgacor1st,INFO SLOT GACOR,8064,2024-11-02T03:42:36+00:00,*BOCORAN GAME ON FIRE & POLA*☝️🤑🤑\n\n*LINK DFT...,wd,...,True,homoglyph|number_substitution,4,id,perlu_review,unlabeled,,Perdana-2024 keyword seed + Telegram platform ...,seed,2026-07-09T17:14:22+00:00
2,telegram,public_channel_api,infoslotgacor1st,judol_candidate,infoslotgacor1st,INFO SLOT GACOR,8062,2024-10-29T20:45:59+00:00,BOCORAN GAME ON FIRE & POLA☝️🤑🤑\n\nLINK DFTR/L...,wd,...,True,homoglyph|number_substitution,4,id,perlu_review,unlabeled,,Perdana-2024 keyword seed + Telegram platform ...,seed,2026-07-09T17:14:22+00:00
3,telegram,public_channel_api,infoslotgacor1st,judol_candidate,infoslotgacor1st,INFO SLOT GACOR,8059,2024-10-26T04:45:41+00:00,SELAMAT PAGI MENJELANG SIANG BREKU SEMUA🤑🤑🤑\nY...,wd,...,True,homoglyph|number_substitution,4,id,perlu_review,unlabeled,,Perdana-2024 keyword seed + Telegram platform ...,seed,2026-07-09T17:14:22+00:00
4,telegram,public_channel_api,infoslotgacor1st,judol_candidate,infoslotgacor1st,INFO SLOT GACOR,8058,2024-10-25T07:00:48+00:00,JUM'AT BERKAH BREKUUU🤑🤑🤑\nHidup adalah permain...,wd,...,True,number_substitution,4,id,perlu_review,unlabeled,,Perdana-2024 keyword seed + Telegram platform ...,seed,2026-07-09T17:14:22+00:00
5,telegram,public_channel_api,infoslotgacor1st,judol_candidate,infoslotgacor1st,INFO SLOT GACOR,8056,2024-10-25T03:39:31+00:00,BOCORAN GAME ON FIRE & POLA☝️🤑🤑\n\nLINK DFTR/L...,wd,...,True,homoglyph|number_substitution,4,id,perlu_review,unlabeled,,Perdana-2024 keyword seed + Telegram platform ...,seed,2026-07-09T17:14:22+00:00
6,telegram,public_channel_api,infoslotgacor1st,judol_candidate,infoslotgacor1st,INFO SLOT GACOR,8055,2024-10-22T22:05:45+00:00,BOCORAN GAME ON FIRE SUDAH TURUN☝️🤑🤑\n\nLINK D...,wd,...,True,homoglyph|number_substitution,4,id,perlu_review,unlabeled,,Perdana-2024 keyword seed + Telegram platform ...,seed,2026-07-09T17:14:22+00:00
7,telegram,public_channel_api,infoslotgacor1st,judol_candidate,infoslotgacor1st,INFO SLOT GACOR,8052,2024-10-21T22:21:00+00:00,ORANG NYA YANG HOKI APA SITUS NYA YANG GACOR B...,gacor|wd,...,True,homoglyph|number_substitution,4,id,perlu_review,unlabeled,,Perdana-2024 keyword seed + Telegram platform ...,seed,2026-07-09T17:14:22+00:00
8,telegram,public_channel_api,infoslotgacor1st,judol_candidate,infoslotgacor1st,INFO SLOT GACOR,8051,2024-10-21T12:22:03+00:00,*LINK LOGIN*\n:\nhttps://anakraja77.org/cfxq\n...,bonus|gacor|maxwin,...,True,homoglyph|number_substitution,4,id,perlu_review,unlabeled,,Perdana-2024 keyword seed + Telegram platform ...,seed,2026-07-09T17:14:22+00:00
9,telegram,public_channel_api,infoslotgacor1st,judol_candidate,infoslotgacor1st,INFO SLOT GACOR,8050,2024-10-20T18:49:38+00:00,BOCORAN GAME ON FIRE SUDAH TURUN🤑🤑🤑\n\nLINK DF...,wd,...,True,homoglyph|number_substitution,4,id,perlu_review,unlabeled,,Perdana-2024 keyword seed + Telegram platform ...,seed,2026-07-09T17:14:22+00:00


## 9. Audit Summary

Use these checks before deciding whether to scale collection or adjust seed channels.

In [12]:
if len(df_raw):
    summary = {
        "rows": len(df_raw),
        "channels": int(df_raw["channel_username"].nunique()) if "channel_username" in df_raw else 0,
        "keyword_candidates": int(df_raw["candidate_flag_keyword"].sum()) if "candidate_flag_keyword" in df_raw else 0,
        "contact_candidates": int(df_raw["candidate_flag_contact"].sum()) if "candidate_flag_contact" in df_raw else 0,
        "unicode_candidates": int(df_raw["candidate_flag_unicode"].sum()) if "candidate_flag_unicode" in df_raw else 0,
        "source_groups": df_raw["source_group"].value_counts(dropna=False).to_dict() if "source_group" in df_raw else {},
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    display(df_raw[["channel_username", "source_group", "candidate_score", "matched_keywords", "raw_text"]].head(10))
else:
    print("Run collection first, then rerun this audit cell.")

{
  "rows": 1403,
  "channels": 8,
  "keyword_candidates": 440,
  "contact_candidates": 257,
  "unicode_candidates": 1034,
  "source_groups": {
    "control_negative": 825,
    "judol_candidate": 578
  }
}


,channel_username,source_group,candidate_score,matched_keywords,raw_text
0,infoslotgacor1st,judol_candidate,4,pragmatic,MULAI SAAT INI PRGMATIC BISA MAIN DI BET 400P ...
1,infoslotgacor1st,judol_candidate,4,wd,*BOCORAN GAME ON FIRE & POLA*☝️🤑🤑\n\n*LINK DFT...
2,infoslotgacor1st,judol_candidate,4,wd,BOCORAN GAME ON FIRE & POLA☝️🤑🤑\n\nLINK DFTR/L...
3,infoslotgacor1st,judol_candidate,4,wd,SELAMAT PAGI MENJELANG SIANG BREKU SEMUA🤑🤑🤑\nY...
4,infoslotgacor1st,judol_candidate,4,wd,JUM'AT BERKAH BREKUUU🤑🤑🤑\nHidup adalah permain...
5,infoslotgacor1st,judol_candidate,4,wd,BOCORAN GAME ON FIRE & POLA☝️🤑🤑\n\nLINK DFTR/L...
6,infoslotgacor1st,judol_candidate,4,wd,BOCORAN GAME ON FIRE SUDAH TURUN☝️🤑🤑\n\nLINK D...
7,infoslotgacor1st,judol_candidate,4,gacor|wd,ORANG NYA YANG HOKI APA SITUS NYA YANG GACOR B...
8,infoslotgacor1st,judol_candidate,4,bonus|gacor|maxwin,*LINK LOGIN*\n:\nhttps://anakraja77.org/cfxq\n...
9,infoslotgacor1st,judol_candidate,4,wd,BOCORAN GAME ON FIRE SUDAH TURUN🤑🤑🤑\n\nLINK DF...


## 10. Reset Local Telegram Scrape State

Only run this when you intentionally want to delete the local Telegram scrape database and session output CSVs.

In [10]:
RESET_TELEGRAM_SCRAPE = False

if RESET_TELEGRAM_SCRAPE:
    conn.close()
    for path in [DB_FILE, RAW_OUTPUT, ANNOTATION_OUTPUT, CHANNEL_OUTPUT]:
        if path.exists():
            path.unlink()
            print(f"Deleted {path}")
    print("Reset complete. Rerun from the SQLite setup cell.")
else:
    print("Reset skipped. Set RESET_TELEGRAM_SCRAPE=True only when you really want to delete local scrape state.")

Reset skipped. Set RESET_TELEGRAM_SCRAPE=True only when you really want to delete local scrape state.
